In [1]:
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

import pandas as pd
import numpy as np
import pickle

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"

train_df = pd.read_parquet(PROCESSED_DIR / "train.parquet")

with open(PROCESSED_DIR / "encoders.pkl", "rb") as f:
    encoders = pickle.load(f)
user_to_idx = encoders["user_to_idx"]
movie_to_idx = encoders["movie_to_idx"]
idx_to_movie = {v: k for k, v in movie_to_idx.items()}

movies = pd.read_parquet(PROCESSED_DIR / "movies_with_idx.parquet")

# ALS factors
user_factors_df = pd.read_parquet(MODELS_DIR / "als_user_factors.parquet")
item_factors_df = pd.read_parquet(MODELS_DIR / "als_item_factors.parquet")
user_factors = {row["id"]: np.array(row["features"]) for _, row in user_factors_df.iterrows()}
item_factors = {row["id"]: np.array(row["features"]) for _, row in item_factors_df.iterrows()}

# Popularity fallback
movie_popularity_order = train_df["movie_idx"].value_counts().index.tolist()

# Per-user seen movies (to exclude from recs)
train_seen = train_df.groupby("user_idx")["movie_idx"].apply(set)

# Cold-start threshold (from Phase 7 findings)
train_counts_per_user = train_df.groupby("user_idx").size()
COLD_START_THRESHOLD = 50

print(f"Users: {len(user_factors)}, Movies w/ factors: {len(item_factors)}, Total movies: {len(movie_to_idx)}")
movies.head()

Users: 6040, Movies w/ factors: 3667, Total movies: 3706


,movie_id,title,genres,movie_idx,genres_str
0,1,Toy Story (1995),Animation|Children's|Comedy,0,Animation Children's Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy,1,Adventure Children's Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance,2,Comedy Romance
3,4,Waiting to Exhale (1995),Comedy|Drama,3,Comedy Drama
4,5,Father of the Bride Part II (1995),Comedy,4,Comedy


In [2]:
def get_recommendations(user_id, n=10):
    """
    Given a raw user_id (MovieLens ID), return top-N movie recommendations.
    Uses ALS for users with enough history; falls back to popularity
    for cold-start users or unknown users, per Phase 7 findings.
    """
    # Unknown user entirely (not in training data) -> pure popularity fallback
    if user_id not in user_to_idx:
        top_movie_idxs = movie_popularity_order[:n]
        strategy = "popularity_fallback (unknown user)"
    else:
        user_idx = user_to_idx[user_id]
        n_ratings = train_counts_per_user.get(user_idx, 0)
        seen = train_seen.get(user_idx, set())

        if n_ratings <= COLD_START_THRESHOLD or user_idx not in user_factors:
            # Cold-start user -> popularity fallback, excluding seen movies
            top_movie_idxs = [m for m in movie_popularity_order if m not in seen][:n]
            strategy = "popularity_fallback (cold-start)"
        else:
            # Regular user -> ALS scoring
            u_vec = user_factors[user_idx]
            scores = {
                m_idx: np.dot(u_vec, i_vec)
                for m_idx, i_vec in item_factors.items()
                if m_idx not in seen
            }
            top_movie_idxs = [m for m, s in sorted(scores.items(), key=lambda x: -x[1])[:n]]
            strategy = "als"

    # Map back to movie details
    result = movies[movies["movie_idx"].isin(top_movie_idxs)][["movie_id", "title", "genres"]].copy()
    result["movie_idx"] = result["movie_id"].map(movie_to_idx)
    result = result.set_index("movie_idx").loc[top_movie_idxs].reset_index(drop=True)

    return result, strategy


# Test on a regular (active) user
test_recs, test_strategy = get_recommendations(user_id=1, n=10)
print(f"Strategy used: {test_strategy}\n")
test_recs

Strategy used: popularity_fallback (cold-start)



,movie_id,title,genres
0,2858,American Beauty (1999),Comedy|Drama
1,1196,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Drama|Sci-Fi|War
2,1210,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Romance|Sci-Fi|War
3,589,Terminator 2: Judgment Day (1991),Action|Sci-Fi|Thriller
4,480,Jurassic Park (1993),Action|Adventure|Sci-Fi
5,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller
6,1198,Raiders of the Lost Ark (1981),Action|Adventure
7,593,"Silence of the Lambs, The (1991)",Drama|Thriller
8,1580,Men in Black (1997),Action|Adventure|Comedy|Sci-Fi
9,110,Braveheart (1995),Action|Drama|War


In [3]:
# Find a user with lots of ratings to test the ALS path
most_active_user_idx = train_counts_per_user.idxmax()
idx_to_user = {v: k for k, v in user_to_idx.items()}
most_active_user_id = idx_to_user[most_active_user_idx]

print(f"Most active user: user_id={most_active_user_id}, n_ratings={train_counts_per_user[most_active_user_idx]}")

test_recs2, test_strategy2 = get_recommendations(user_id=most_active_user_id, n=10)
print(f"Strategy used: {test_strategy2}\n")
test_recs2

Most active user: user_id=4169, n_ratings=1852
Strategy used: als



,movie_id,title,genres
0,2931,Time of the Gypsies (Dom za vesanje) (1989),Drama
1,2503,"Apple, The (Sib) (1998)",Drama
2,2776,"Marcello Mastroianni: I Remember Yes, I Rememb...",Documentary
3,3092,Chushingura (1962),Drama
4,1664,Nénette et Boni (1996),Drama
5,53,Lamerica (1994),Drama
6,557,Mamma Roma (1962),Drama
7,811,"Bewegte Mann, Der (1994)",Comedy
8,1148,"Wrong Trousers, The (1993)",Animation|Comedy
9,1901,Dear Jesse (1997),Documentary


In [4]:
# A user_id that doesn't exist in the dataset at all
test_recs3, test_strategy3 = get_recommendations(user_id=999999, n=10)
print(f"Strategy used: {test_strategy3}\n")
test_recs3

Strategy used: popularity_fallback (unknown user)



,movie_id,title,genres
0,2858,American Beauty (1999),Comedy|Drama
1,1196,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Drama|Sci-Fi|War
2,260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Fantasy|Sci-Fi
3,1210,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Romance|Sci-Fi|War
4,2028,Saving Private Ryan (1998),Action|Drama|War
5,589,Terminator 2: Judgment Day (1991),Action|Sci-Fi|Thriller
6,480,Jurassic Park (1993),Action|Adventure|Sci-Fi
7,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller
8,1270,Back to the Future (1985),Comedy|Sci-Fi
9,1198,Raiders of the Lost Ark (1981),Action|Adventure


In [5]:
# Normalize popularity rank into a 0-1 boost score
movie_pop_counts = train_df["movie_idx"].value_counts()
max_pop = movie_pop_counts.max()
popularity_score = (movie_pop_counts / max_pop).to_dict()

def get_recommendations_v2(user_id, n=10, popularity_weight=0.3):
    """
    v2: blends ALS score with a popularity boost to avoid overly obscure recs.
    """
    if user_id not in user_to_idx:
        top_movie_idxs = movie_popularity_order[:n]
        strategy = "popularity_fallback (unknown user)"
    else:
        user_idx = user_to_idx[user_id]
        n_ratings = train_counts_per_user.get(user_idx, 0)
        seen = train_seen.get(user_idx, set())

        if n_ratings <= COLD_START_THRESHOLD or user_idx not in user_factors:
            top_movie_idxs = [m for m in movie_popularity_order if m not in seen][:n]
            strategy = "popularity_fallback (cold-start)"
        else:
            u_vec = user_factors[user_idx]
            # Normalize ALS scores to 0-1 range before blending
            raw_scores = {
                m_idx: np.dot(u_vec, i_vec)
                for m_idx, i_vec in item_factors.items()
                if m_idx not in seen
            }
            min_s, max_s = min(raw_scores.values()), max(raw_scores.values())
            range_s = max_s - min_s if max_s != min_s else 1

            blended_scores = {
                m_idx: (popularity_weight * popularity_score.get(m_idx, 0)) +
                       ((1 - popularity_weight) * (s - min_s) / range_s)
                for m_idx, s in raw_scores.items()
            }
            top_movie_idxs = [m for m, s in sorted(blended_scores.items(), key=lambda x: -x[1])[:n]]
            strategy = "als_popularity_blend"

    result = movies[movies["movie_idx"].isin(top_movie_idxs)][["movie_id", "title", "genres"]].copy()
    result["movie_idx"] = result["movie_id"].map(movie_to_idx)
    result = result.set_index("movie_idx").loc[top_movie_idxs].reset_index(drop=True)
    return result, strategy


test_recs4, test_strategy4 = get_recommendations_v2(user_id=4169, n=10)
print(f"Strategy used: {test_strategy4}\n")
test_recs4

Strategy used: als_popularity_blend



,movie_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,3114,Toy Story 2 (1999),Animation|Children's|Comedy
2,2931,Time of the Gypsies (Dom za vesanje) (1989),Drama
3,2503,"Apple, The (Sib) (1998)",Drama
4,1148,"Wrong Trousers, The (1993)",Animation|Comedy
5,745,"Close Shave, A (1995)",Animation|Comedy|Thriller
6,2776,"Marcello Mastroianni: I Remember Yes, I Rememb...",Documentary
7,2355,"Bug's Life, A (1998)",Animation|Children's|Comedy
8,39,Clueless (1995),Comedy|Romance
9,3175,Galaxy Quest (1999),Adventure|Comedy|Sci-Fi


In [6]:
import pickle

recommendation_artifacts = {
    "user_to_idx": user_to_idx,
    "movie_to_idx": movie_to_idx,
    "user_factors": user_factors,
    "item_factors": item_factors,
    "movie_popularity_order": movie_popularity_order,
    "popularity_score": popularity_score,
    "train_seen": train_seen.to_dict(),
    "train_counts_per_user": train_counts_per_user.to_dict(),
    "cold_start_threshold": COLD_START_THRESHOLD,
}

with open(MODELS_DIR / "recommendation_artifacts.pkl", "wb") as f:
    pickle.dump(recommendation_artifacts, f)

movies.to_parquet(PROCESSED_DIR / "movies_final.parquet", index=False)

print("Saved recommendation_artifacts.pkl and movies_final.parquet")
print(f"File size: {(MODELS_DIR / 'recommendation_artifacts.pkl').stat().st_size / 1024 / 1024:.2f} MB")

Saved recommendation_artifacts.pkl and movies_final.parquet
File size: 3.32 MB
